In [ ]:
# FOLDERS
import os
# NOTE: cluster-specific path, replace with your own location
PATH = '/leonardo_scratch/large/userexternal/ndebole0' 
PATH = '/leonardo_scratch/fast/IscrC_ARGO/' 

# Folder containing Linear Probes with entire dataset (LP-ALL)
LPALL_PATH = os.path.join(PATH, 'migrated_results/LPALL/normalized')
LPAT_PATH = os.path.join(PATH, 'migrated_results/LPAT/normalized')

In [ ]:
import os
LP_PATH = os.path.join(PATH, 'migrated_results')
CLIP_SCORES_PATH = os.path.join(PATH, 'migrated_results/2026_02_23_11_49-CLIPSCORES.json')

In [ ]:
import pandas as pd
# NOTE: cluster-specific path, replace with your own location
df_argo = pd.read_csv('/leonardo/home/userexternal/ndebole0/projects/argus/argus/figsdino/results.csv')


lp = df_argo[(df_argo['model'] == 'LP')].copy()
lp["method"] = r"\texttt{LP}-@"

argo_active = df_argo[(df_argo['model'] == 'MCSVGP')].copy()

argo_active = df_argo[(df_argo['model'] == 'MCSVGP') & (df_argo['acq_fn'] == 'activeours')].copy()
argo_active["method"] = r"ACTIVE \texttt{VH-CBM}-@"
argo_random = df_argo[(df_argo['model'] == 'MCSVGP') & (df_argo['acq_fn'] == 'randomc')].copy()
argo_random["method"] = r"RAND \texttt{VH-CBM}-@"


In [ ]:
# NOTE: cluster-specific path, replace with your own location
df_argocbm = pd.read_csv('/leonardo/home/userexternal/ndebole0/projects/argus/argus/figs/cbmat.csv')
df_argocbm['method'] = r"\texttt{CBM}-@"
df_argocbm['backbone'] = 'ViT-L/14'

# create duplicate and modify backbone
df_dino = df_argocbm.copy()
df_dino['backbone'] = 'dinov3'

# combine into a single dataframe
df_combined_cbmat = pd.concat([df_argocbm, df_dino], ignore_index=True)

In [ ]:
import os
pr = pd.DataFrame()
seeds = [42, 43, 44, 45, 46]

for data in ["celeba", "shapes3d", "dermamnist", "cub"]:
    for seed in seeds:
        tmp = pd.read_json(f"{LP_PATH}/probes_full_ds_{data}.json")
        tmp["dataset"] = data
        tmp["method"] = r"\texttt{LP-All}"
        tmp["backbone"] = 'ViT-L/14'
        pr = pd.concat([pr, tmp], axis=0)

lpall = pd.DataFrame()
lpall_files = os.listdir(LPALL_PATH)
for file in lpall_files:
    dataset = file.split("_")[1]
    tmp = pd.read_json(f"{LPALL_PATH}/{file}")
    tmp["dataset"] = dataset
    tmp["method"] = r"\texttt{LP-All}"
    lpall = pd.concat([lpall, tmp], axis=0)

lpall = pd.concat([lpall, pr])

lpat = pd.DataFrame()
lpat_files = os.listdir(LPAT_PATH)
for file in lpat_files:
    dataset = file.split("-")[1].split("_")[1]
    tmp = pd.read_json(f"{LPAT_PATH}/{file}")
    tmp["dataset"] = dataset
    tmp["method"] = r"\texttt{LP}-@"
    lpat = pd.concat([lpat, tmp], axis=0)


In [ ]:
clip = pd.read_json(CLIP_SCORES_PATH)
clip["method"] = r"\texttt{CLIP}"

In [ ]:
df = pd.concat([argo_active, argo_random, lpat, df_combined_cbmat], axis=0)

sizes = {
    "celeba": 39*25000,
    "shapes3d": 5*48000,
    "dermamnist": 7007,
    "cub": 112*4800
}

df["train_size"] = df["dataset"].apply(lambda x: sizes[x])
df["concept_used_perc"] = df["concept_used"].div(df["train_size"])

print(df.method.unique())

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sns
from matplotlib.pyplot import subplots
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
        'figure.dpi': 300,  # high resolution
        'savefig.dpi': 300,
        'axes.titlesize': 18,  # title font size
        'axes.labelsize': 18,  # x/y label font size
        'xtick.labelsize': 18,  # tick label sizes
        'ytick.labelsize': 18,
        'legend.fontsize': 14,
        # 'font.size': 14
    })
sns.set_theme(
            style="ticks",
            palette="colorblind",
            rc={
                "text.usetex": True,
                'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{amsmath} \usepackage{bm}',
                "font.family": "serif",
            })

# x= concept_used, y = metric che vogliamo (f1_raw)
dict_names = {"celeba": r"\texttt{CelebA}",
              "shapes3d": r"\texttt{Shapes3D}",
              "dermamnist": r"\texttt{DermaMNIST}",
              "cub": r"\texttt{CUB}"
              }
markers_dict = {
    r"ACTIVE \texttt{VH-CBM}-@": "o",
    r"RAND \texttt{VH-CBM}-@": "d",
    r"\texttt{LP}-@": "s",
    r"\texttt{CBM}-@": "^",



}
text_dataset_pos_dict = {
    "celeba": (0.15, 0.87),
    "shapes3d": (0.2, 0.87),
    "dermamnist": (0.23, 0.87),
    "cub": (0.1, 0.87)
}
dataset_filter_dict = {
    "celeba": 0.015,
    "shapes3d": .008,
    "dermamnist": .055,
    "cub": .08
}

metrics = ["f1_raw"]
datasets = ["shapes3d", "celeba", "dermamnist", "cub"]

for backbone in ['ViT-L/14', 'dinov3']:
    fig, ax = plt.subplots(1, 4, figsize=(20,4), sharey=True)
    for i, dataset in enumerate(datasets):
        tmp = df[(df["backbone"] == backbone) & (df["dataset"] == dataset) & (df["concept_used_perc"]< dataset_filter_dict[dataset])].copy()
        sns.lineplot(tmp, x="concept_used_perc", y="f1_raw", hue="method", hue_order=list(markers_dict.keys()),
        ax= ax[i], style="method", markers=markers_dict, markersize=10)
        avg = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].mean()
        std = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].std()
        ax[i].axhline(avg, color="black", linestyle="--", label="\\texttt{LP-All}")
        ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        if backbone in ['dinov3']:
            pass
        else:
            avg_clip = clip[clip["dataset"] == dataset]["f1_raw"].mean()
            std_clip = clip[clip["dataset"] == dataset]["f1_raw"].std()
            ax[i].axhline(avg_clip, color="grey", linestyle="-.", label="\\texttt{CLIP}")
        # ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        if i != 1:
            ax[i].get_legend().remove()
        else:
            handles, labels = ax[i].get_legend_handles_labels()
            fig.legend(handles, labels, title=None, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.03), fontsize=18)
            ax[i].get_legend().remove()
            # ax[i].legend(title=None, loc="lower right", fontsize=22, ncol=1)
        if i == 0:
            ax[i].set_ylabel(r"\ensuremath{F_1(C)}", fontsize=20)
        ax[i].grid(axis='y', alpha=0.3)
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax[i].text(text_dataset_pos_dict[dataset][0], text_dataset_pos_dict[dataset][1], dict_names[dataset], transform=ax[i].transAxes, ha="center", va="center", bbox= props, fontsize=18)
        ax[i].set_xlabel("")

    fig.text(0.5, -.01, r"\% Concepts Annotations", ha="center", va="center", fontsize=20)
    for a in ax:
        a.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.1%}"))

    # for a in ax:
    #     a.xaxis.set_major_formatter(FuncFormatter(strip_leading_zero3))
    plt.subplots_adjust(wspace=.06)  # Adjust the width space between subplots
    plt.show()
    plt.close()

    b = backbone.replace('/',"")
    fig.savefig(os.path.join(PATH, f"plot-{b}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(PATH, f"plot-{b}.png"), bbox_inches="tight")


In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sns
from matplotlib.pyplot import subplots
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
        'figure.dpi': 300,  # high resolution
        'savefig.dpi': 300,
        'axes.titlesize': 18,  # title font size
        'axes.labelsize': 18,  # x/y label font size
        'xtick.labelsize': 18,  # tick label sizes
        'ytick.labelsize': 18,
        'legend.fontsize': 14,
        # 'font.size': 14
    })
sns.set_theme(
            style="ticks",
            palette="colorblind",
            rc={
                "text.usetex": True,
                'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{amsmath} \usepackage{bm}',
                "font.family": "serif",
            })

# x= concept_used, y = metric che vogliamo (f1_raw)
dict_names = {"celeba": r"\texttt{CelebA}",
              "shapes3d": r"\texttt{Shapes3D}",
              "dermamnist": r"\texttt{DermaMNIST}",
              "cub": r"\texttt{CUB}"
              }
markers_dict = {
    #r"ACTIVE \texttt{VH-CBM}-@": "o",
    r"RAND \texttt{VH-CBM}-@": "d",
    r"\texttt{LP}-@": "s",
    r"\texttt{CBM}-@": "^",



}
text_dataset_pos_dict = {
    "celeba": (0.15, 0.87),
    "shapes3d": (0.2, 0.87),
    "dermamnist": (0.23, 0.87),
    "cub": (0.1, 0.87)
}
dataset_filter_dict = {
    "celeba": 0.015,
    "shapes3d": .008,
    "dermamnist": .055,
    "cub": .08
}
palette_dict = {
    #r"ACTIVE \texttt{VH-CBM}-@": "steelblue",
    r"RAND \texttt{VH-CBM}-@": "#009E73",
    r"\texttt{LP}-@": "darkorange",
    r"\texttt{CBM}-@": "#CC79A7",
}

metrics = ["f1_raw"]
datasets = ["shapes3d", "celeba", "dermamnist", "cub"]

for backbone in ['ViT-L/14', 'dinov3']:
    fig, ax = plt.subplots(1, 4, figsize=(20,4), sharey=True)
    for i, dataset in enumerate(datasets):
        tmp = df[(df["backbone"] == backbone) & (df["dataset"] == dataset) & (df["concept_used_perc"]< dataset_filter_dict[dataset]) & (df["method"] != r"ACTIVE \texttt{VH-CBM}-@")].copy()
        sns.lineplot(tmp, x="concept_used_perc", y="f1_raw", hue="method", palette=palette_dict, hue_order=list(markers_dict.keys()),
        ax= ax[i], style="method", markers=markers_dict, markersize=10)
        avg = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].mean()
        std = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].std()
        ax[i].axhline(avg, color="black", linestyle="--", label="\\texttt{LP-All}")
        ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        if backbone in ['dinov3']:
            pass
        else:
            avg_clip = clip[clip["dataset"] == dataset]["f1_raw"].mean()
            std_clip = clip[clip["dataset"] == dataset]["f1_raw"].std()
            ax[i].axhline(avg_clip, color="grey", linestyle="-.", label="\\texttt{CLIP}")
        # ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        if i != 1:
            ax[i].get_legend().remove()
        else:
            handles, labels = ax[i].get_legend_handles_labels()
            label_remap = {
                #r"ACTIVE \texttt{VH-CBM}-@": r"\texttt{VH-CBM}-@",
                r"RAND \texttt{VH-CBM}-@": r"\texttt{VH-CBM}-@",
            }
            labels = [label_remap.get(l, l) for l in labels]
            if backbone != 'dinov3':
                fig.legend(handles, labels, title=None, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.03), fontsize=18)
            ax[i].get_legend().remove()
            # ax[i].legend(title=None, loc="lower right", fontsize=22, ncol=1)
        if i == 0:
            ax[i].set_ylabel(r"\ensuremath{F_1(C)}", fontsize=20)
        ax[i].grid(axis='y', alpha=0.3)
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax[i].text(text_dataset_pos_dict[dataset][0], text_dataset_pos_dict[dataset][1], dict_names[dataset], transform=ax[i].transAxes, ha="center", va="center", bbox= props, fontsize=18)
        ax[i].set_xlabel("")

    fig.text(0.5, -.01, r"\% Concepts Annotations", ha="center", va="center", fontsize=20)
    for a in ax:
        a.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.1%}"))

    # for a in ax:
    #     a.xaxis.set_major_formatter(FuncFormatter(strip_leading_zero3))
    plt.subplots_adjust(wspace=.06)  # Adjust the width space between subplots
    plt.show()
    plt.close()

    b = backbone.replace('/',"")
    fig.savefig(os.path.join(PATH, f"plot-{b}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(PATH, f"plot-{b}.png"), bbox_inches="tight")


In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sns
from matplotlib.pyplot import subplots
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
        'figure.dpi': 300,  # high resolution
        'savefig.dpi': 300,
        'axes.titlesize': 18,  # title font size
        'axes.labelsize': 18,  # x/y label font size
        'xtick.labelsize': 18,  # tick label sizes
        'ytick.labelsize': 18,
        'legend.fontsize': 14,
        # 'font.size': 14
    })
sns.set_theme(
            style="ticks",
            palette="colorblind",
            rc={
                "text.usetex": True,
                'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{amsmath} \usepackage{bm}',
                "font.family": "serif",
            })

# x= concept_used, y = metric che vogliamo (f1_raw)
dict_names = {"celeba": r"\texttt{CelebA}",
              "shapes3d": r"\texttt{Shapes3D}",
              "dermamnist": r"\texttt{DermaMNIST}",
              "cub": r"\texttt{CUB}"
              }
markers_dict = {
    r"ACTIVE \texttt{VH-CBM}-@": "o",
    r"RAND \texttt{VH-CBM}-@": "d",
    #r"\texttt{LP}-@": "s",
    #r"\texttt{CBM}-@": "^",



}
text_dataset_pos_dict = {
    "celeba": (0.15, 0.87),
    "shapes3d": (0.2, 0.87),
    "dermamnist": (0.23, 0.87),
    "cub": (0.1, 0.87)
}
dataset_filter_dict = {
    "celeba": 0.015,
    "shapes3d": .008,
    "dermamnist": .055,
    "cub": .08
}
palette_dict = {
    r"ACTIVE \texttt{VH-CBM}-@": "steelblue",
    r"RAND \texttt{VH-CBM}-@": "#009E73",
}

metrics = ["f1_raw"]
datasets = ["shapes3d", "celeba", "dermamnist", "cub"]

for backbone in ['ViT-L/14', 'dinov3']:
    fig, ax = plt.subplots(1, 4, figsize=(20,4), sharey=False)
    for i, dataset in enumerate(datasets):
        tmp = df[(df["backbone"] == backbone) & (df["dataset"] == dataset) & (df["concept_used_perc"]< dataset_filter_dict[dataset])].copy()
        tmp = tmp[tmp["method"].str.contains("ACTIVE|RAND", na=False)]
        
        sns.lineplot(tmp, x="concept_used_perc", y="f1_raw", hue="method", palette=palette_dict, hue_order=list(markers_dict.keys()),
        ax= ax[i], style="method", markers=markers_dict, markersize=10)
        avg = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].mean()
        std = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].std()
        #ax[i].axhline(avg, color="black", linestyle="--", label="\\texttt{LP-All}")
        #ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        # ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        if i != 0:
            ax[i].set_ylabel("")
            #ax[i].tick_params(labelleft=False)
        if i != 1:
            ax[i].get_legend().remove()
        else:
            handles, labels = ax[i].get_legend_handles_labels()
            if backbone != 'dinov3':
                fig.legend(handles, labels, title=None, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.03), fontsize=18)
            ax[i].get_legend().remove()
            # ax[i].legend(title=None, loc="lower right", fontsize=22, ncol=1)
        if i == 0:
            ax[i].set_ylabel(r"\ensuremath{F_1(C)}", fontsize=20)
        ax[i].grid(axis='y', alpha=0.3)
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax[i].text(text_dataset_pos_dict[dataset][0], text_dataset_pos_dict[dataset][1], dict_names[dataset], transform=ax[i].transAxes, ha="center", va="center", bbox= props, fontsize=18)
        ax[i].set_xlabel("")

    fig.text(0.5, -.01, r"\% Concepts Annotations", ha="center", va="center", fontsize=20)
    for a in ax:
        a.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.1%}"))

    # for a in ax:
    #     a.xaxis.set_major_formatter(FuncFormatter(strip_leading_zero3))
    plt.subplots_adjust(wspace=.26)  # Adjust the width space between subplots
    plt.show()
    plt.close()

    b = backbone.replace('/',"")
    fig.savefig(os.path.join(PATH, f"plot-{b}-ar.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(PATH, f"plot-{b}-ar.png"), bbox_inches="tight")


In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sns
from matplotlib.pyplot import subplots
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
        'figure.dpi': 300,  # high resolution
        'savefig.dpi': 300,
        'axes.titlesize': 18,  # title font size
        'axes.labelsize': 18,  # x/y label font size
        'xtick.labelsize': 18,  # tick label sizes
        'ytick.labelsize': 18,
        'legend.fontsize': 14,
        # 'font.size': 14
    })
sns.set_theme(
            style="ticks",
            palette="colorblind",
            rc={
                "text.usetex": True,
                'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{amsmath} \usepackage{bm}',
                "font.family": "serif",
            })

# x= concept_used, y = metric che vogliamo (f1_raw)
dict_names = {"celeba": r"\texttt{CelebA}",
              "shapes3d": r"\texttt{Shapes3D}",
              "dermamnist": r"\texttt{DermaMNIST}",
              "cub": r"\texttt{CUB}"
              }
markers_dict = {
    r"ACTIVE \texttt{VH-CBM}-@": "o",
    r"RAND \texttt{VH-CBM}-@": "d",
    #r"\texttt{LP}-@": "s",
    #r"\texttt{CBM}-@": "^",



}
text_dataset_pos_dict = {
    "celeba": (0.15, 0.87),
    "shapes3d": (0.2, 0.87),
    "dermamnist": (0.23, 0.87),
    "cub": (0.1, 0.87)
}
dataset_filter_dict = {
    "celeba": 0.015,
    "shapes3d": .008,
    "dermamnist": .055,
    "cub": .08
}

metrics = ["f1_raw"]
datasets = ["shapes3d", "celeba", "dermamnist", "cub"]
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=False)
bbtr = {
    'ViT-L/14':"CLIP", 
    'dinov3':"DINO"}

for b_idx, backbone in enumerate(['ViT-L/14', 'dinov3']):
    ax = axes[b_idx]
    
    # Add backbone label on the left
    fig.text(0.07, 1 - (b_idx + 0.5) / len(bbtr), bbtr[backbone], 
             va='center', ha='right', fontsize=20, fontweight='bold',
             transform=fig.transFigure)
    
    for i, dataset in enumerate(datasets):
        tmp = df[(df["backbone"] == backbone) & (df["dataset"] == dataset) & (df["concept_used_perc"]< dataset_filter_dict[dataset])].copy()
        tmp = tmp[tmp["method"].str.contains("ACTIVE|RAND", na=False)]
        
        sns.lineplot(tmp, x="concept_used_perc", y="f1_raw", hue="method", hue_order=list(markers_dict.keys()),
        ax= ax[i], style="method", markers=markers_dict, markersize=10)
        avg = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].mean()
        std = lpall[(lpall["dataset"] == dataset) & (lpall["backbone"] == backbone)]["f1_raw"].std()
        #ax[i].axhline(avg, color="black", linestyle="--", label="\\texttt{LP-All}")
        #ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        # ax[i].fill_between(ax[i].get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        if i != 0:
            ax[i].set_ylabel("")
            #ax[i].tick_params(labelleft=False)
        if i != 1:
            ax[i].get_legend().remove()
        else:
            handles, labels = ax[i].get_legend_handles_labels()
            fig.legend(handles, labels, title=None, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.03), fontsize=18)
            ax[i].get_legend().remove()
            # ax[i].legend(title=None, loc="lower right", fontsize=22, ncol=1)
        if i == 0:
            ax[i].set_ylabel(r"\ensuremath{F_1(C)}", fontsize=20)
        ax[i].grid(axis='y', alpha=0.3)
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax[i].text(text_dataset_pos_dict[dataset][0], text_dataset_pos_dict[dataset][1], dict_names[dataset], transform=ax[i].transAxes, ha="center", va="center", bbox= props, fontsize=18)
        ax[i].set_xlabel("")

    fig.text(0.5, -.01, r"\% Concepts Annotations", ha="center", va="center", fontsize=20)
    for a in ax:
        a.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.1%}"))

   # Outside the loop:
fig.text(0.5, -.01, r"\% Concepts Annotations", ha="center", va="center", fontsize=20)
plt.subplots_adjust(wspace=0.26, hspace=0.4)
fig.savefig(os.path.join(PATH, "plot-combined-ar.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(PATH, "plot-combined-ar.png"), bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sns
from matplotlib.pyplot import subplots
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
        'figure.dpi': 300,  # high resolution
        'savefig.dpi': 300,
        'axes.titlesize': 18,  # title font size
        'axes.labelsize': 18,  # x/y label font size
        'xtick.labelsize': 18,  # tick label sizes
        'ytick.labelsize': 18,
        'legend.fontsize': 14,
        # 'font.size': 14
    })
sns.set_theme(
            style="ticks",
            palette="colorblind",
            rc={
                "text.usetex": True,
                'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{amsmath} \usepackage{bm}',
                "font.family": "serif",
            })

# x= concept_used, y = metric che vogliamo (f1_raw)
dict_names = {"celeba": r"\texttt{CelebA}",
              "shapes3d": r"\texttt{Shapes3D}",
              "dermamnist": r"\texttt{DermaMNIST}",
              "cub": r"\texttt{CUB}"
              }
markers_dict = {
    r"ACTIVE \texttt{VH-CBM}-@": "o",
    r"RAND \texttt{VH-CBM}-@": "d",
    r"\texttt{LP}-@": "s",
    r"\texttt{CBM}-@": "^",



}
text_dataset_pos_dict = {
    "celeba": (0.15, 0.87),
    "shapes3d": (0.2, 0.87),
    "dermamnist": (0.23, 0.87),
    "cub": (0.1, 0.87)
}
dataset_filter_dict = {
    "celeba": 0.015,
    "shapes3d": .008,
    "dermamnist": .055,
    "cub": .08
}

metrics = ["f1_raw"]
datasets = ["shapes3d", "celeba", "dermamnist", "cub"]

for backbone in ['ViT-L/14', 'dinov3']:
    fig, ax = plt.subplots(2, 2, figsize=(10, 8), sharey=True)
    for i, dataset in enumerate(datasets):
        curr_ax = ax[i//2, i%2]  # convert flat index to 2D
        tmp = df[(df["backbone"] == backbone) & (df["dataset"] == dataset) & (df["concept_used_perc"] < dataset_filter_dict[dataset])].copy()
        sns.lineplot(tmp, x="concept_used_perc", y="f1_raw", hue="method", hue_order=list(markers_dict.keys()),
                    ax=curr_ax, style="method", markers=markers_dict, markersize=10)
        avg = pr[pr["dataset"] == dataset]["f1_raw"].mean()
        std = pr[pr["dataset"] == dataset]["f1_raw"].std()
        curr_ax.axhline(avg, color="black", linestyle="--", label="\\texttt{LP-All}")
        curr_ax.fill_between(curr_ax.get_xlim(), avg - std, avg + std, color="gray", alpha=0.2)
        avg_clip = clip[clip["dataset"] == dataset]["f1_raw"].mean()
        std_clip = clip[clip["dataset"] == dataset]["f1_raw"].std()
        curr_ax.axhline(avg_clip, color="grey", linestyle="-.", label="\\texttt{CLIP}")
        if i != 1:
            curr_ax.get_legend().remove()
        else:
            handles, labels = curr_ax.get_legend_handles_labels()
            fig.legend(handles, labels, title=None, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.06), fontsize=14)
            #fig.legend(handles, labels, title=None, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.03), fontsize=14)
            curr_ax.get_legend().remove()
        if i % 2 == 0:  # left column only
            curr_ax.set_ylabel(r"\ensuremath{F_1(C)}", fontsize=16)
        else:
            curr_ax.set_ylabel("")
        curr_ax.grid(axis='y', alpha=0.3)
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        curr_ax.text(text_dataset_pos_dict[dataset][0], text_dataset_pos_dict[dataset][1], dict_names[dataset],
                    transform=curr_ax.transAxes, ha="center", va="center", bbox=props, fontsize=14)
        curr_ax.set_xlabel("")

    fig.text(0.5, -.01, r"\% Concepts Annotations", ha="center", va="center", fontsize=16)

    for a in ax.flat:
        a.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.1%}"))

    plt.tight_layout()
    plt.subplots_adjust(top=0.95)  # lower value = more space for legend, plots move down
    b = backbone.replace('/',"")
    fig.savefig(os.path.join(PATH, f"plot-2x2-{b}.pdf"), bbox_inches="tight")


In [ ]:
import os, itertools, subprocess, datetime

DATASET = 'shapes3d'
FAST_STORAGE = os.environ["SCRATCH"]
to_evaluate = []
# Experiment combinations
BASE = f"{os.environ['SCRATCH']}/results/ARGOCBM/dinov3"
models = os.listdir(BASE)
for folder in models:
    #print(folder)
    if "-340-" not in folder and "-510-" not in folder:
        continue
    #    print(folder)
    if DATASET not in folder:
        continue
    
    metrics_path = os.path.join(BASE,folder,"metrics.txt")
    to_evaluate.append(os.path.join(BASE,folder))
    #if os.path.exists(metrics_path):
    #    continue
    #else:
    #    if os.path.exists(os.path.join(FAST_STORAGE,"results/ARGOCBM",folder,"args.txt")): 
    #        to_evaluate.append(os.path.join(FAST_STORAGE,"results/ARGOCBM",folder))

print(len(to_evaluate))